# 📞 Customer Support AI QA Evaluator
### *End-to-End GenAI Evaluation Pipeline with LangChain, Pydantic & Dynamic Routing*

---

## 📌 Overview & Architecture
This notebook implements a production-grade **LLM-as-a-Judge** pipeline to automatically evaluate customer support call transcripts.

### 🔄 Pipeline Stages:
1. **LLM Abstraction Layer**: Multi-provider initialization (`OpenAI` or `Google Gemini`) via unified LangChain interfaces.
2. **Intent Classification**: Classify call transcripts into categories (`billing`, `claims`, `complaint`, `general_query`) with **Pydantic Schema Validation**.
3. **Dynamic Routing**: Route calls conditionally to specialized evaluation criteria based on their classification.
4. **Modular Evaluators**: Single-responsibility chains evaluating:
   - **Tone & Empathy** (politeness, customer acknowledgment)
   - **Resolution Quality** (problem-solving, next steps)
   - **Knowledge Accuracy** (correctness, clarity)
5. **Hierarchical QA Synthesis**: Synthesizes granular evaluation outputs into an executive summary and actionable coaching recommendations (`FinalReport`).
6. **Batch Processing & Export**: Resilient dataset evaluation and persistence to Excel.

```
┌─────────────────────┐
│ Raw Call Transcript │
└──────────┬──────────┘
           ▼
┌─────────────────────────────────┐
│ Stage 1: Classification Chain   │ ──► [Predicted Call Type]
└─────────────────────────────────┘
           ▼
┌─────────────────────────────────┐
│ Stage 2: Dynamic Routing Logic  │ ──► [Evaluation Plan]
└──────────┬──────────────────────┘
           ├─────────────────┬─────────────────┐
           ▼                 ▼                 ▼
     ┌───────────┐     ┌───────────┐     ┌───────────┐
     │ Tone &    │     │ Knowledge │     │Resolution │
     │  Empathy  │     │ Accuracy  │     │  Quality  │
     └─────┬─────┘     └─────┬─────┘     └─────┬─────┘
           └─────────────────┼─────────────────┘
                             ▼
┌──────────────────────────────────────────────┐
│ Stage 4: Hierarchical Reporting Chain        │ ──► [Summary & Recommendations]
└──────────────────────────────────────────────┘
```

---

## 1️⃣ Environment Setup, Configuration & Multi-Provider LLM Initialization

### 🧠 Core LangChain Concepts:
- **`BaseChatModel` (`ChatOpenAI`, `ChatGoogleGenerativeAI`)**: Unified model interface abstracting vendor-specific API variations.
- **Factory Pattern (`load_llm`)**: Decouples model provider selection (`LLM_PROVIDER` in `.env`) from downstream logic, enabling zero-code provider switching.
- **`temperature=0.3`**: Low temperature ensures objective, deterministic, and reproducible evaluation scores.

In [1]:
# =============================================================================
# Step 1: Import Core Libraries
# =============================================================================
import os
import json
import pandas as pd
from dotenv import load_dotenv

# LangChain Chat Model Wrappers
from langchain_openai import ChatOpenAI
from langchain_google_genai import ChatGoogleGenerativeAI

# =============================================================================
# Step 2: Load Environment Variables & System Configuration
# =============================================================================
# Load API keys (OPENAI_API_KEY, GEMINI_API_KEY) and LLM_PROVIDER from .env
load_dotenv()

# Load centralized configuration (model names, temperature, labels, criteria)
with open("config/config.json", "r") as f:
    config = json.load(f)

# =============================================================================
# Step 3: Load Call Transcripts Dataset
# =============================================================================
df = pd.read_csv("data/transcripts.csv")
print("✅ Data Loaded successfully")
print(df.head(), "\n")

# =============================================================================
# Step 4: Multi-Provider LLM Factory
# =============================================================================
def load_llm(config):
    """
    Factory function to instantiate the selected LLM provider dynamically.
    
    Parameters:
        config (dict): Parsed config.json dictionary containing model hyperparameters.
    Returns:
        BaseChatModel: An instance of ChatOpenAI or ChatGoogleGenerativeAI.
    """
    provider = os.getenv("LLM_PROVIDER", "gemini").lower()
    print(f"🔹 Selected LLM Provider: {provider}")

    if provider == "openai":
        return ChatOpenAI(
            model=config["llm"]["openai_model"],
            temperature=config["llm"]["temperature"]
        )
    elif provider == "gemini":
        return ChatGoogleGenerativeAI(
            model=config["llm"]["gemini_model"],
            temperature=config["llm"]["temperature"]
        )
    else:
        raise ValueError(f"Unsupported LLM_PROVIDER: '{provider}'. Supported: 'openai', 'gemini'")

# Instantiate the configured LLM
llm = load_llm(config)
print(f"✅ LLM Initialized: {llm.__class__.__name__}")

# =============================================================================
# Step 5: Smoke Test / Health Check
# =============================================================================
# .invoke() sends a prompt to the LLM and returns an AIMessage object
response = llm.invoke("Say 'setup successful' in one short sentence.")
print("\n🧠 LLM Health Check Response:")
print(response.content)

✅ Data Loaded successfully
   call_id agent_name                                         transcript  \
0        1      Alice  Customer: I was charged twice for my premium t...   
1        2        Bob  Customer: My claim has been pending for 2 week...   
2        3    Charlie  Customer: This is the third time I'm calling! ...   
3        4      Diana  Customer: Can you explain my coverage details?...   

  expected_call_type  
0            billing  
1             claims  
2          complaint  
3      general_query   

🔹 Selected LLM Provider: gemini


d:\projects\python\stepIntoAI\customer_support_AI_pipeline\lngcncsp\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


✅ LLM Initialized: ChatGoogleGenerativeAI

🧠 LLM Health Check Response:
[{'type': 'text', 'text': 'The setup was successful.', 'extras': {'signature': 'EtwNCtkNARFNMg+CtP+0/OD6g9EbYGLdE3T85KHyBYg68ekhXUG0BVGVx/1Vzr/i2jaBNhXwnOBtRMr0tvXWp3VTTV8VYchGCC8erJzv/JP0bKZM8RMSXNb4zv+1wbfnRXATUNtHimdJUkjVcP9F/1ehKqcGq1Vkki78jnXZEuFnWxZE/Q0x41V3UROZtNE0dOTFy/Sm1grTAhyitPFLgJ0qtIfatDFQQn/oEiiZdvhmV6Z2Ya6aizGchTdu4Su1B30ZmmXC+YVE3Id0HU+89JzBcMA4De3mO68+C8bc3odyc1Zj/do7xXGZs+/pTVshpT5KTZ9RSYS/X3nyD+XobXS2Qj37Pgd/bJIDw3/1zZZgVO7FLzMl1ygBc7jfMjamSZLtxoaMhkBWQAXhCxI4EHHGrZmkN9E1k9Revdr4eoi6C5jt1hyoULB+WGYmBudcKfsYPlRpr8YYCvkGi8x8lxp1/roWWpeCd/gGFdp10DITRbc1LH77cDgCrHV4eHfncFtARxr4QyZC9u/vIDD4pG54H1gRBytRU9nVIU3uavi8qC+9Af0ykn26dHbxFrdI6MrnRdmkPkW4RAxR65Y5wiPOsIn2d7uAyxj1Rk5TMIaJB1Pbx03R5VPi1K9Fs4VW9OdoISAHYDw2SrxumKg9ncEBYiLFGAXfgTxj6G+M1VbJ18dmDbFQe4EaItFYA1oj9KCXMmDGgAiiXqmOVo1dz88fhe0WW7ipuuGUAbdTFUQOedNyuDlS22c4KR0dnMzU6GjX0y+O6ID1rJpvxPN06404MlqpJhDGCyd3NWV6Xc3pO9gYbuvXa/dpZiO+mLHt3Rn15P2Rehls3sct

## 2️⃣ Intent Classification with Pydantic Schema Enforcement

### 🧠 Core LangChain Concepts:
- **`pydantic.BaseModel` & `Field`**: Defines the strict output schema with data types and descriptions.
- **`PydanticOutputParser`**: Generates format instructions (`parser.get_format_instructions()`) for the prompt and parses raw LLM text into a typed Pydantic object.
- **`PromptTemplate` with `partial_variables`**:
  - `input_variables=["transcript"]`: Dynamic input passed per invocation.
  - `partial_variables`: Pre-filled static parameters (`format_instructions`, `labels` from config) bound at chain creation time.
- **LCEL (LangChain Expression Language) `|`**:
  - `classification_chain = prompt | llm | parser` forms a `RunnableSequence`:
  - `dict` input $\rightarrow$ `PromptTemplate` $\rightarrow$ `PromptValue` $\rightarrow$ `ChatModel` $\rightarrow$ `AIMessage` $\rightarrow$ `PydanticOutputParser` $\rightarrow$ `ClassificationOutput` object.

In [3]:
# =============================================================================
# Step 1: Imports for Structured Output & Prompt Engineering
# =============================================================================
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field

# =============================================================================
# Step 2: Define Output Schema with Pydantic
# =============================================================================
class ClassificationOutput(BaseModel):
    """Schema defining the structured output for call classification."""
    call_type: str = Field(description="Type of customer call (e.g. billing, claims, complaint, general_query)")
    confidence: float = Field(description="Confidence score between 0.0 and 1.0")

# Instantiate parser targeting our Pydantic schema
parser = PydanticOutputParser(pydantic_object=ClassificationOutput)

# =============================================================================
# Step 3: Construct Prompt Template with Partial Variables
# =============================================================================
prompt = PromptTemplate(
    template="""
You are a call classification assistant.

Classify the following customer support transcript into one of these categories:
{labels}

Transcript:
{transcript}

{format_instructions}
""",
    input_variables=["transcript"], # Variable provided at runtime
    partial_variables={              # Variables pre-bound at build time
        "format_instructions": parser.get_format_instructions(),
        "labels": config["classification"]["labels"]
    }
)

# =============================================================================
# Step 4: Assemble LCEL Chain (Prompt | LLM | Parser)
# =============================================================================
classification_chain = prompt | llm | parser

# =============================================================================
# Step 5: Test Chain on a Single Sample
# =============================================================================
sample_text = df.iloc[0]["transcript"]
result = classification_chain.invoke({"transcript": sample_text})

print("✅ Classification Result (Pydantic Object):")
print(f"Call Type: {result.call_type} | Confidence: {result.confidence}")
print(result)

f:\deepak\projects\langchain_essentials\customer_support_AI_pipeline\lngcncsp\Lib\site-packages\langchain_google_genai\chat_models.py:3719: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


✅ Classification Result:
call_type='billing' confidence=0.98


## 3️⃣ Batch Classification with Resilient Error Handling

### 🧠 Key Engineering Practices:
- **Fault Tolerance (`try...except`)**: In production batch pipelines, wrapping individual LLM calls in exception handlers prevents a single network timeout or parsing issue from halting the entire job.
- **`tqdm` Progress Tracking**: Real-time visibility into batch execution rate and progress.
- **DataFrame Integration**: Merging structured predictions back to the original dataset for auditing.

In [4]:
from tqdm import tqdm

results = []

# Iterate through each transcript in the dataset
for i, row in tqdm(df.iterrows(), total=len(df), desc="Classifying Calls"):
    try:
        output = classification_chain.invoke({
            "transcript": row["transcript"]
        })
        
        results.append({
            "call_id": row["call_id"],
            "predicted_call_type": output.call_type,
            "confidence": output.confidence
        })
        
    except Exception as e:
        print(f"❌ Error at row {i} (call_id: {row['call_id']}): {e}")
        results.append({
            "call_id": row["call_id"],
            "predicted_call_type": None,
            "confidence": None
        })

# Convert results to DataFrame and merge with original data
results_df = pd.DataFrame(results)
df = df.merge(results_df, on="call_id")

print("\n✅ Batch Classification Completed")
print(df[["call_id", "expected_call_type", "predicted_call_type", "confidence"]])

## 4️⃣ Dynamic Routing Layer (Conditional Evaluation Plan)

### 🧠 Concept & Benefits:
- **Context-Aware Evaluation**: Not every call needs every metric. E.g., complaints need **Tone & Empathy** checks, whereas general queries need **Knowledge Accuracy** checks.
- **Cost & Latency Optimization**: Routing avoids running unnecessary LLM evaluator chains, drastically saving tokens and execution time.

In [5]:
# =============================================================================
# Step 1: Define Dynamic Routing Function
# =============================================================================
def route_call(call_type):
    """
    Maps predicted call category to a targeted list of evaluation criteria.
    
    Parameters:
        call_type (str): Predicted category (billing, claims, complaint, general_query)
    Returns:
        list[str]: Evaluation dimensions to execute
    """
    if call_type == "billing":
        return ["knowledge_accuracy", "resolution_quality"]
    elif call_type == "claims":
        return ["knowledge_accuracy", "resolution_quality"]
    elif call_type == "complaint":
        return ["tone_empathy", "resolution_quality"]
    elif call_type == "general_query":
        return ["knowledge_accuracy"]
    else:
        return ["knowledge_accuracy"]  # Fallback default

# =============================================================================
# Step 2: Apply Routing Across Dataset
# =============================================================================
df["evaluation_plan"] = df["predicted_call_type"].apply(route_call)

print("✅ Dynamic Routing Plan Applied:")
print(df[["call_id", "predicted_call_type", "evaluation_plan"]])

## 5️⃣ Modular Specialized Evaluator Chains

### 🧠 Design Patterns:
- **Single Responsibility Principle (SRP)**: Breaking evaluation into distinct chains (`tone_chain`, `resolution_chain`, `knowledge_chain`) increases evaluation fidelity compared to one monolithic evaluation prompt.
- **Chain-of-Thought (CoT) Grounding**: Each schema enforces both `score: int` (1-5) and `reasoning: str` explaining why that score was awarded, which forces the LLM to justify its evaluation before finalizing the score.

In [6]:
# =============================================================================
# Evaluator 1: Tone & Empathy Evaluation Chain
# =============================================================================
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field

# 1. Schema: Tone & Empathy
class ToneEvaluation(BaseModel):
    score: int = Field(description="Score between 1 (poor/rude) and 5 (highly empathetic & professional)")
    reasoning: str = Field(description="Detailed justification for the assigned score")

tone_parser = PydanticOutputParser(pydantic_object=ToneEvaluation)

# 2. Prompt Template with specific evaluation rubrics
tone_prompt = PromptTemplate(
    template="""
You are a QA evaluator for customer support calls.

Evaluate the agent's tone and empathy in the following transcript.

Consider:
- Did the agent acknowledge the customer's issue?
- Was the tone polite and professional?
- Did the agent show empathy?

Transcript:
{transcript}

{format_instructions}
""",
    input_variables=["transcript"],
    partial_variables={
        "format_instructions": tone_parser.get_format_instructions()
    }
)

# 3. Assemble Tone Chain
tone_chain = tone_prompt | llm | tone_parser

# 4. Test on a complaint call
sample_text = df[df["predicted_call_type"] == "complaint"].iloc[0]["transcript"]
result = tone_chain.invoke({"transcript": sample_text})

print("✅ Tone Evaluation Result:")
print(f"Score: {result.score}/5")
print(f"Reasoning: {result.reasoning}")

✅ Tone Evaluation Result:
score=3 reasoning="The agent acknowledged the customer's issue by apologizing for the inconvenience, which indicates some level of recognition. However, the response was quite brief and lacked a more personalized touch or deeper empathy. The tone was polite and professional, but it could have been more empathetic by expressing understanding of the customer's frustration due to repeated calls."


In [7]:
# =============================================================================
# Evaluator 2: Resolution Quality Evaluation Chain
# =============================================================================
# 1. Schema: Resolution Quality
class ResolutionEvaluation(BaseModel):
    score: int = Field(description="Score between 1 (unresolved) and 5 (completely resolved with clear next steps)")
    reasoning: str = Field(description="Detailed explanation of the resolution assessment")

resolution_parser = PydanticOutputParser(pydantic_object=ResolutionEvaluation)

# 2. Prompt Template
resolution_prompt = PromptTemplate(
    template="""
You are a QA evaluator for customer support calls.

Evaluate the resolution quality of the agent.

Consider:
- Did the agent fully resolve the customer's issue?
- Were next steps clearly communicated?
- Did the agent confirm resolution before ending?

Transcript:
{transcript}

{format_instructions}
""",
    input_variables=["transcript"],
    partial_variables={
        "format_instructions": resolution_parser.get_format_instructions()
    }
)

# 3. Assemble Resolution Chain
resolution_chain = resolution_prompt | llm | resolution_parser

# 4. Test on a sample transcript
sample_text = df.iloc[0]["transcript"]
result = resolution_chain.invoke({"transcript": sample_text})

print("✅ Resolution Evaluation Result:")
print(f"Score: {result.score}/5")
print(f"Reasoning: {result.reasoning}")

✅ Resolution Evaluation Result:
score=2 reasoning="The agent did not fully resolve the customer's issue as the transcript ends without confirming the resolution or providing next steps. There is no indication that the agent addressed the double charge or offered a solution."


In [8]:
# =============================================================================
# Evaluator 3: Knowledge Accuracy Evaluation Chain
# =============================================================================
# 1. Schema: Knowledge Accuracy
class KnowledgeEvaluation(BaseModel):
    score: int = Field(description="Score between 1 (incorrect/misleading) and 5 (highly accurate & clear)")
    reasoning: str = Field(description="Explanation of the accuracy score")

knowledge_parser = PydanticOutputParser(pydantic_object=KnowledgeEvaluation)

# 2. Prompt Template with instructions for incomplete transcripts
knowledge_prompt = PromptTemplate(
    template="""
You are a QA evaluator for customer support calls.

Evaluate the agent's knowledge accuracy and clarity.

Consider:
- Did the agent provide correct and relevant information?
- Was the explanation clear and easy to understand?
- Did the agent avoid vague or misleading statements?

IMPORTANT:
- If the transcript does not contain enough information, give a moderate score (2 or 3) and explain why.

Transcript:
{transcript}

{format_instructions}
""",
    input_variables=["transcript"],
    partial_variables={
        "format_instructions": knowledge_parser.get_format_instructions()
    }
)

# 3. Assemble Knowledge Chain
knowledge_chain = knowledge_prompt | llm | knowledge_parser

print("✅ Knowledge Chain Ready")

✅ Knowledge Chain Ready


## 6️⃣ Evaluation Orchestrator (Executing Dynamic Plan)

### 🧠 Concept:
- `run_evaluations()` dynamically inspects `eval_plan` for each transcript and conditionally invokes only the required evaluator chains.
- Uses `.model_dump()` on the resulting Pydantic instances to convert them into JSON-serializable Python dictionaries.

In [9]:
# =============================================================================
# Step 1: Define Conditional Execution Runner
# =============================================================================
def run_evaluations(transcript, eval_plan):
    """
    Dynamically executes only the evaluation chains specified in eval_plan.
    
    Parameters:
        transcript (str): Customer support call dialogue.
        eval_plan (list[str]): Criteria to evaluate (e.g. ['tone_empathy', 'resolution_quality']).
    Returns:
        dict: Aggregated dictionary of evaluation scores and reasoning.
    """
    results = {}

    if "tone_empathy" in eval_plan:
        try:
            tone_result = tone_chain.invoke({"transcript": transcript})
            results["tone"] = tone_result.model_dump()
        except Exception as e:
            results["tone"] = {"error": str(e)}

    if "knowledge_accuracy" in eval_plan:
        try:
            knowledge_result = knowledge_chain.invoke({"transcript": transcript})
            results["knowledge"] = knowledge_result.model_dump()
        except Exception as e:
            results["knowledge"] = {"error": str(e)}

    if "resolution_quality" in eval_plan:
        try:
            resolution_result = resolution_chain.invoke({"transcript": transcript})
            results["resolution"] = resolution_result.model_dump()
        except Exception as e:
            results["resolution"] = {"error": str(e)}

    return results

# =============================================================================
# Step 2: Apply Evaluation Runner to Dataset
# =============================================================================
evaluation_outputs = []

for i, row in tqdm(df.iterrows(), total=len(df), desc="Running Evaluations"):
    output = run_evaluations(row["transcript"], row["evaluation_plan"])
    evaluation_outputs.append({
        "call_id": row["call_id"],
        "evaluation_output": output
    })

# Convert to DataFrame and merge
eval_df = pd.DataFrame(evaluation_outputs)
df = df.merge(eval_df, on="call_id")

print("\n✅ Evaluation Completed for All Calls")

# Display structured output of the first evaluated call
import pprint
pprint.pprint(df.iloc[0]["evaluation_output"])

Running Evaluations: 100%|██████████| 4/4 [00:10<00:00,  2.63s/it]


✅ Evaluation Completed

{'knowledge': {'reasoning': "The agent's response to the customer's issue is "
                            'incomplete as the transcript does not provide any '
                            'information about the resolution or details '
                            'regarding the double charge. While the agent '
                            'initiated the process to check the issue, there '
                            'is no evidence of correct or relevant information '
                            'being provided, nor is there clarity in the '
                            'explanation. Therefore, a moderate score is '
                            'appropriate.',
               'score': 3},
 'resolution': {'reasoning': "The agent did not fully resolve the customer's "
                             'issue as the transcript ends abruptly without '
                             'confirming the resolution or providing next '
                             'steps. There is no 

## 7️⃣ Hierarchical QA Synthesis (Managerial Reporting Chain)

### 🧠 Concept & Architecture:
- **Multi-Stage / Hierarchical Reasoning**: Instead of feeding raw text directly to a manager prompt, we feed the **structured outputs of lower-level evaluator chains** (`evaluation_output`).
- **Actionable Coaching Output**: Generates an executive summary and a list of concrete coaching recommendations (`list[str]`) for supervisor review.

In [10]:
# =============================================================================
# Step 1: Schema for Final Managerial QA Report
# =============================================================================
class FinalReport(BaseModel):
    """Schema for synthesized coaching report."""
    summary: str = Field(description="Overall concise performance summary of the agent")
    recommendations: list[str] = Field(description="List of practical, actionable coaching recommendations")

final_parser = PydanticOutputParser(pydantic_object=FinalReport)

# =============================================================================
# Step 2: Synthesis Prompt Template
# =============================================================================
final_prompt = PromptTemplate(
    template="""
You are a QA manager reviewing customer support calls.

Based on the evaluation results below, generate:
1. A concise summary of the agent's performance
2. A list of actionable recommendations for improvement

Evaluation Data:
{evaluation_output}

IMPORTANT:
- Be specific and practical
- Do not repeat raw scores
- Focus on actionable agent coaching

{format_instructions}
""",
    input_variables=["evaluation_output"],
    partial_variables={
        "format_instructions": final_parser.get_format_instructions()
    }
)

# =============================================================================
# Step 3: Assemble Final Reporting Chain
# =============================================================================
final_chain = final_prompt | llm | final_parser

# =============================================================================
# Step 4: Test Synthesis on Sample Row
# =============================================================================
sample_eval = df.iloc[0]["evaluation_output"]
result = final_chain.invoke({"evaluation_output": sample_eval})

print("✅ Final QA Report Generated:")
print(f"Summary: {result.summary}\n")
print("Recommendations:")
for r in result.recommendations:
    print(f"- {r}")

✅ Final QA Report:
summary="The agent demonstrated a moderate level of knowledge but failed to provide a complete resolution to the customer's issue regarding a double charge. The conversation lacked clarity and did not include necessary information or next steps, leading to an unsatisfactory customer experience." recommendations=['Enhance training on common billing issues, specifically double charges, to ensure agents can provide accurate information and resolutions.', 'Implement a checklist for agents to follow during calls to ensure all relevant details are covered, including confirming resolution and next steps.', 'Encourage agents to summarize the key points of the conversation and confirm understanding with the customer before ending the call.', 'Provide agents with access to updated knowledge bases and resources to improve their ability to answer customer inquiries effectively.', 'Conduct regular role-playing sessions to practice handling complex customer issues and improve comm

## 8️⃣ Batch Synthesis & Master Dataset Finalization

Generates executive summaries and coaching recommendations for every call in the dataset.

In [11]:
final_outputs = []

# Iterate across evaluated dataset
for i, row in tqdm(df.iterrows(), total=len(df), desc="Generating Final Reports"):
    try:
        result = final_chain.invoke({
            "evaluation_output": row["evaluation_output"]
        })

        final_outputs.append({
            "call_id": row["call_id"],
            "summary": result.summary,
            "recommendations": result.recommendations
        })

    except Exception as e:
        print(f"❌ Error generating report at row {i}: {e}")
        final_outputs.append({
            "call_id": row["call_id"],
            "summary": None,
            "recommendations": None
        })

# Convert to DataFrame and merge
final_df = pd.DataFrame(final_outputs)
df = df.merge(final_df, on="call_id")

print("\n✅ Final Reports Generated Successfully\n")

# Display structured final output
df[["call_id", "predicted_call_type", "evaluation_output", "summary", "recommendations"]]

Generating Final Reports: 100%|██████████| 4/4 [00:13<00:00,  3.35s/it]


✅ Final Reports Generated



,call_id,predicted_call_type,evaluation_output,summary,recommendations
0,1,billing,"{'knowledge': {'score': 3, 'reasoning': 'The a...",The agent demonstrated a moderate level of kno...,[Enhance training on issue resolution to ensur...
1,2,claims,"{'knowledge': {'score': 2, 'reasoning': 'The t...",The agent demonstrated insufficient knowledge ...,[Enhance product knowledge training to ensure ...
2,3,complaint,"{'tone': {'score': 3, 'reasoning': 'The agent ...",The agent demonstrated a polite tone and ackno...,[Enhance empathy in communication by using mor...
3,4,general_query,"{'knowledge': {'score': 3, 'reasoning': 'The t...",The agent demonstrated a willingness to explai...,[Enhance knowledge of coverage details by revi...


## 9️⃣ Classification Accuracy Metric & Excel Export

Computes model performance against labeled ground truth and persists complete QA artifacts.

In [12]:
# Calculate intent classification accuracy against ground truth
accuracy = (df["expected_call_type"] == df["predicted_call_type"]).mean()
print(f"🎯 Classification Accuracy: {accuracy:.2%}")

🎯 Classification Accuracy: 1.00


In [ ]:
# Persist structured dataset to Excel
df.to_excel("data/output.xlsx", index=False)
print("✅ Results successfully saved to 'data/output.xlsx'")